# IOAI — 2025 Stage 1 Hallucination Detection (Colab 자동 설정판)

이 노트북은 IOAI 로컬 연습 사이트에서 **데이터·학습환경이 자동 준비**되도록 생성되었습니다.
아래 **설정 셀을 먼저 실행**하면 공식 GitHub 저장소에서 이 문제 폴더만 부분 클론으로 받아
(전체 6.6GB 가 아니라 해당 폴더만), 그 폴더로 이동한 뒤 이후 셀이 그대로 학습/예측을 합니다.
완료 후 생성되는 제출 파일을 내려받아 연습 사이트의 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** 로 바꾸면 학습이 빨라집니다.

In [ ]:
# === 데이터 + 환경 자동 설정 (가장 먼저 실행) ===
# 공식 공개 저장소에서 이 문제 폴더만 부분 클론(sparse)으로 받고 그 폴더로 이동한다.
import os
REPO_URL = "https://github.com/OlimpiadaAI/II-OlimpiadaAI"
CLONE = "II-OlimpiadaAI"
SUBDIR = "1_etap/2_wykrywanie_halucynacji"
WORKDIR = "1_etap/2_wykrywanie_halucynacji"
# Colab 은 /content 가 홈. 재실행해도 경로가 안정적이도록 고정 기준에서 시작한다.
BASE = "/content" if os.path.isdir("/content") else os.getcwd()
os.chdir(BASE)
if not os.path.isdir(os.path.join(CLONE, SUBDIR)):
    !git clone --filter=blob:none --no-checkout --depth 1 $REPO_URL $CLONE
    !cd $CLONE && git sparse-checkout set "$SUBDIR"
    !cd $CLONE && git checkout
os.chdir(os.path.join(BASE, CLONE, WORKDIR))
print("작업 폴더:", os.getcwd())
print("내용:", sorted(os.listdir(".")))

# Wykrywanie Halucynacji

<img src="https://live.staticflickr.com/65535/54208132682_73767c3560_b.jpg" alt="Embedded Photo" width="500">

*Obraz wygenerowany przy użyciu modelu DALL-E.*

## Wstęp

Modele językowe pomagają nam w codziennych zadaniach, takich jak poprawianie tekstów, pisanie kodu czy odpowiadanie na pytania. 
Są one również coraz częściej wykorzystywane w takich dziedzinach jak medycyna czy edukacja.

Jednak skąd możemy wiedzieć, czy wygenerowane przez nie odpowiedzi są poprawne? Modele językowe nie zawsze posiadają pełną wiedzę na zadany temat, a mimo to mogą formułować odpowiedzi, które brzmią wiarygodnie, lecz w rzeczywistości wprowadzają w błąd. Takie niepoprawne odpowiedzi nazywamy halucynacjami.

## Zadanie

W tym zadaniu zmierzysz się z wykrywaniem halucynacji w odpowiedziach na pytania faktograficzne generowane przez duże modele językowe (LLM).
Przeanalizujesz zbiór danych, który pomoże w ocenie, czy odpowiedzi generowane przez model językowy są faktycznie poprawne, czy zawierają halucynacje.

Każdy przykład w zbiorze danych zawiera:

- **Pytanie** np. "Jaka jest główna odpowiedzialność Departamentu Obrony USA?"
- **Odpowiedź modelu językowego** np. "Główną odpowiedzialnością jest obrona kraju."
- **Tokeny** związane z generacją odpowiedzi.
- **Cztery alternatywne odpowiedzi** wygenerowane z przez ten sam model z większą temperaturą.
- **Tokeny alternatywnych odpowiedzi** wygenerowane z przez ten sam model z większą temperaturą.
- **Prawdopodobieństwa alternatywnych odpowiedzi** wygenerowane z przez ten sam model z większą temperaturą.
- **Etykietę (`is_correct`)** wskazującą, czy główna odpowiedź jest poprawna według zaufanego źródła.


Przykład:
```json
[
    {
        "question_id": 34,
        "question": "What is the name of the low-cost carrier that operates as a wholly owned subsidiary of Singapore Airlines?",
        "answer": "Scoot is the low-cost carrier that operates as a wholly owned subsidiary of Singapore Airlines.",
        "tokens": [" Sco", "ot", " is", ..., " Airlines", ".", "\n"],
        "supporting_answers": [
            "As a wholly owned subsidiary of Singapore Airlines, <answer> Scoot </answer> stands as a low-cost carrier that revolutionized air travel in the region.",
            "Scoot, a subsidiary of <answer> Singapore Airlines </answer> , is the low-cost carrier that operates under the same brand.",
            "<answer> Scoot </answer> is the low-cost carrier that operates as a wholly owned subsidiary of Singapore Airlines.",
            "Singapore Airlines operates a low-cost subsidiary named <answer> Scoot </answer> , offering affordable and efficient air travel options to passengers."
        ],
        "supporting_tokens": [
            [" As", " a", ..., ".", "<answer>"],
            [" Sco", "ot", ..., " brand", ".", "\n"],
            ["<answer>", " Sco", ..., ".", "\n"],
            [" Singapore", " Airlines", ..., ".", "\n"]
        ],
        "supporting_probabilities": [
            [0.0029233775567263365, 0.8621460795402527, ..., 0.018515007570385933],
            [0.42073577642440796, 0.9999748468399048, ..., 0.9166142344474792],
            [0.3258324861526489, 0.9969879984855652, ..., 0.921079695224762],
            [0.11142394691705704, 0.960810661315918, ..., 0.9557166695594788]
        ],
        "is_correct": true
    },
    .
    .
    .
]
```

### Dane
Dane dostępne dla Ciebie w tym zadaniu to:

* `train.json` - zbiór danych zawierający 2967 pytań oraz odpowiedzi.
* `valid.json` - 990 dodatkowych pytań.


### Kryterium Oceny

ROC AUC (ang. *Receiver Operating Characteristic Area Under Curve*) to miara jakości klasyfikatora binarnego. Pokazuje zdolność modelu do odróżniania między dwiema klasami - tutaj halucynacją (false) i poprawną odpowiedzią (true). 

- **ROC (Receiver Operating Characteristic)**: Wykres pokazujący zależność między *True Positive Rate* (czułość) a *False Positive Rate* (1-specyficzność) przy różnych progach decyzyjnych.
- **AUC (Area Under Curve)**: Pole pod wykresem ROC, które przyjmuje wartości od 0 do 1:
  - **1.0**: Model perfekcyjny.
  - **0.5**: Model losowy (brak zdolności do odróżniania klas).

Im wyższa wartość AUC, tym lepiej model radzi sobie z klasyfikacją.

Za to zadanie możesz zdobyć pomiędzy 0 a 100 punktów. Wynik będzie skalowany liniowo w zależności od wartości ROC AUC:

- **ROC AUC ≤ 0.7**: 0 punktów.
- **ROC AUC ≥ 0.82**: 100 punktów.
- **Wartości pomiędzy 0.7 a 0.82**: skalowane liniowo.

Wzór na wynik:  
$$
\text{Punkty} = 
\begin{cases} 
0 & \text{dla } \text{ROC AUC} \leq 0.7 \\
100 \times \frac{\text{ROC AUC} - 0.7}{0.82 - 0.7} & \text{dla } 0.7 < \text{ROC AUC} < 0.82 \\
100 & \text{dla } \text{ROC AUC} \geq 0.82
\end{cases}
$$


## Ograniczenia
* Twoje rozwiazanie będzie testowane na Platformie Konkursowej bez dostępu do internetu oraz w środowisku bez GPU.
* Ewaluacja Twojego finalnego rozwiązania na Platformie Konkursowej nie może trwać dłużej niż 5 minut bez GPU.
* Lista dopuszczalnych bibliotek: `xgboost`, `scikit-learn`, `numpy`, `pandas`, `matplotlib`.


## Pliki Zgłoszeniowe
Ten notebook uzupełniony o Twoje rozwiązanie (patrz funkcja `predict_hallucinations`).

## Ewaluacja
Pamiętaj, że podczas sprawdzania flaga `FINAL_EVALUATION_MODE` zostanie ustawiona na `True`.

Za to zadanie możesz zdobyć pomiędzy 0 a 100 punktów. Liczba punktów, którą zdobędziesz, będzie wyliczona na (tajnym) zbiorze testowym na Platformie Konkursowej na podstawie wyżej wspomnianego wzoru, zaokrąglona do liczby całkowitej. Jeśli Twoje rozwiązanie nie będzie spełniało powyższych kryteriów lub nie będzie wykonywać się prawidłowo, otrzymasz za zadanie 0 punktów.


# Kod Startowy
W tej sekcji inicjalizujemy środowisko poprzez zaimportowanie potrzebnych bibliotek i funkcji. Przygotowany kod ułatwi Tobie efektywne operowanie na danych i budowanie właściwego rozwiązania.

In [ ]:
######################### NIE ZMIENIAJ TEJ KOMÓRKI PODCZAS WYSYŁANIA ##########################

FINAL_EVALUATION_MODE = False  # W czasie sprawdzania twojego rozwiązania, zmienimy tą wartość na True

import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn as sk
from sklearn.metrics import roc_auc_score
import xgboost as xgb
import shutil

def download_data(train=("1TGEDaxw4GKfSq0fpqSk0wRpUSc8GgZN0", "train.json"),
                  valid=("1qrr7bZk6Uct8DeC-V8Bc1qD5su56ryFd", "valid.json")):
    """Pobiera zbiór danych z Google Drive i zapisuje go w folderze 'data'."""
    import gdown
    
    # Utwórz lub zresetuj folder 'data'
    if not os.path.exists('data'):
        os.makedirs('data')
    else:
        shutil.rmtree('data')
        os.makedirs('data')

    GDRIVE_DATA = [train, valid]
    
    for file_id, file_name in GDRIVE_DATA:        
        # Pobierz plik z Google Drive i zapisz go w folderze 'data'
        url = f'https://drive.google.com/uc?id={file_id}'
        output = f'data/{file_name}'
        gdown.download(url, output, quiet=False)
        
        print(f"Downloaded: {file_name}")

# Pobierz dane tylko jeśli nie jesteś w trybie FINAL_EVALUATION_MODE
if not FINAL_EVALUATION_MODE:
    download_data()


## Ładowanie Danych
Za pomocą poniższego kodu dane zostaną wczytane i odpowiednio przygotowane.

In [ ]:
######################### NIE ZMIENIAJ TEJ KOMÓRKI PODCZAS WYSYŁANIA ##########################

def load_data(folder='data'):
    # Wczytaj dane z plików
    train_path = os.path.join(folder, 'train.json')
    valid_path = os.path.join(folder, 'valid.json')
    
    with open(train_path, 'r') as f:
        train = json.load(f)
    with open(valid_path, 'r') as f:
        valid = json.load(f)

    return train, valid

train, valid = load_data("data")

print(json.dumps(train[0], indent=2))

print(f"\nWszystkie przykłady treningowe: {len(train)}")
print(f"Wszystkie przykłady walidacyjne: {len(valid)}")

## Kod z Kryterium Oceniającym

Kod, zbliżony do poniższego, będzie używany do oceny rozwiązania na zbiorze testowym.

In [ ]:
######################### NIE ZMIENIAJ TEJ KOMÓRKI PODCZAS WYSYŁANIA ##########################

def compute_score(roc_auc: float) -> float:
    """
    Oblicza wynik punktowy na podstawie wartości ROC AUC.

    :param roc_auc: Wartość float w zakresie [0.0, 1.0]
    :return: Wynik punktowy zgodny z określoną funkcją
    """
    if roc_auc <= 0.7:
        return 0
    elif 0.7 < roc_auc < 0.82:
        return int(round(100 * (roc_auc - 0.7) / (0.82 - 0.7)))
    else:
        return 100


def evaluate_algorithm(dataset, algorithm, verbose=False):
    """
    Ewaluacja algorytmu wykrywania halucynacji na podanym zbiorze danych.

    Parametry
    ----------
    dataset : list
        Oznaczony zbiór danych, gdzie każdy element to słownik zawierający klucz 'is_correct'.
    algorithm : callable
        Funkcja, która przyjmuje pojedynczy przykład (słownik) i zwraca prawdopodobieństwo halucynacji.
    verbose : bool
        Jeśli True, wypisuje dodatkowe informacje dla każdego przykładu oraz podsumowanie.

    Zwraca
    -------
    roc_auc : float
        Wartość pola pod krzywą ROC (ROC AUC) dla predykcji.
    """
    predicted_ys = [] # Lista przechowująca przewidywane prawdopodobieństwa halucynacji

    for i, entry in enumerate(dataset):
        # Tworzenie kopii próbki i usunięcie etykiety, aby uzyskać dane wejściowe bez oznaczeń
        sample_unlabeled = dict(entry)
        sample_unlabeled.pop('is_correct', None)

        try:
            # Przewidywanie prawdopodobieństwa dla pojedynczej próbki
            pred_prob = algorithm(sample_unlabeled)
            predicted_ys.append(pred_prob)

        except Exception as e:
            # Jeśli wystąpi błąd, domyślnie ustawiamy prawdopodobieństwo na 0.5
            predicted_ys.append(0.5)
            if verbose:
                print(f"Sample {i} => Error: {e}")

    predicted_ys = np.array(predicted_ys, dtype=np.float32)
    ys = []
    for entry in dataset:
        ys.append(1 if entry.get('is_correct') else 0)
    ys = np.array(ys, dtype=np.int32)
    
    # Obliczenie metryki ROC AUC
    roc_auc = roc_auc_score(ys, predicted_ys)

    # Obliczenie końcowego wyniku na podstawie ROC AUC
    points = compute_score(roc_auc)

    if verbose:
        print(f"\nLiczba próbek: {len(dataset)}")
        print(f"ROC AUC: {roc_auc:.4f}")
        print(f"Wynik punktowy: {points}")

    return points

# Twoje Rozwiązanie
W tej sekcji należy umieścić Twoje rozwiązanie. Wprowadzaj zmiany wyłącznie tutaj!

In [ ]:
# TODO: Użyj danych treningowych, aby stworzyć tutaj model lub algorytm.

def predict_hallucinations(sample):
    # TODO: Uruchom swój model lub algorytm na tym zestawie danych.
    # TODO: Zwróć listę prawdopodobieństw dla każdego przykładu w zestawie danych.
    
    prediction = 0.5
    return prediction


# Ewaluacja

Uruchomienie poniższej komórki pozwoli sprawdzić, ile punktów zdobyłoby Twoje rozwiązanie na danych walidacyjnych. Przed wysłaniem upewnij się, że cały notebook wykonuje się od początku do końca bez błędów i bez konieczności ingerencji użytkownika po wybraniu opcji "Run All".

In [ ]:
if not FINAL_EVALUATION_MODE:
    roc_auc = evaluate_algorithm(valid, predict_hallucinations, verbose=True)

Podczas sprawdzania model zostanie zapisany jako `your_model.pkl` i oceniony na zbiorze testowym.

In [ ]:
######################### NIE ZMIENIAJ TEJ KOMÓRKI PODCZAS WYSYŁANIA ##########################
if FINAL_EVALUATION_MODE:      
    import cloudpickle
      
    OUTPUT_PATH = "file_output"
    FUNCTION_FILENAME = "your_model.pkl"
    FUNCTION_OUTPUT_PATH = os.path.join(OUTPUT_PATH, FUNCTION_FILENAME)

    if not os.path.exists(OUTPUT_PATH):
        os.makedirs(OUTPUT_PATH)

    with open(FUNCTION_OUTPUT_PATH, "wb") as f:
        cloudpickle.dump(predict_hallucinations, f)

## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.csv', 'submission.zip', 'submission.jsonl', 'submission.json']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)